In [1]:
from sklearn.preprocessing import OrdinalEncoder

import pandas as pd
import numpy as np
from pathlib import Path
from utils.extraction_utils import extract_from_list_column, to_float_cm, to_float_kg
from utils.analizing_utils import print_nan_stat_in_row, calc_outliers, remove_outliers
from configuration.configuration import load_config

In [2]:
path = Path()

<h1>Wczytanie danych</h1>

In [3]:
origin_df = pd.read_csv(path / 'data' / 'superheroes_data.csv')
origin_df

,id,name,intelligence,strength,speed,durability,power,combat,full-name,alter-egos,...,race,height,weight,eye-color,hair-color,occupation,base,group-affiliation,relatives,url
0,1,A-Bomb,38.0,100.0,17.0,80.0,24.0,64.0,Richard Milhouse Jones,No alter egos found.,...,Human,"[""6'8"", '203 cm']","['980 lb', '441 kg']",Yellow,No Hair,"Musician, adventurer, author; formerly talk sh...",-,"Hulk Family; Excelsior (sponsor), Avengers (ho...",Marlo Chandler-Jones (wife); Polly (aunt); Mrs...,https://www.superherodb.com/pictures2/portrait...
1,2,Abe Sapien,88.0,28.0,35.0,65.0,100.0,85.0,Abraham Sapien,No alter egos found.,...,Icthyo Sapien,"[""6'3"", '191 cm']","['145 lb', '65 kg']",Blue,No Hair,Paranormal Investigator,-,Bureau for Paranormal Research and Defense,"Edith Howard (wife, deceased)",https://www.superherodb.com/pictures2/portrait...
2,3,Abin Sur,50.0,90.0,53.0,64.0,99.0,65.0,NaN,No alter egos found.,...,Ungaran,"[""6'1"", '185 cm']","['200 lb', '90 kg']",Blue,No Hair,"Green Lantern, former history professor",Oa,"Green Lantern Corps, Black Lantern Corps","Amon Sur (son), Arin Sur (sister), Thaal Sines...",https://www.superherodb.com/pictures2/portrait...
3,4,Abomination,63.0,80.0,53.0,90.0,62.0,95.0,Emil Blonsky,No alter egos found.,...,Human / Radiation,"[""6'8"", '203 cm']","['980 lb', '441 kg']",Green,No Hair,Ex-Spy,Mobile,former member of the crew of the Andromeda Sta...,"Nadia Dornova Blonsky (wife, separated)",https://www.superherodb.com/pictures2/portrait...
4,5,Abraxas,88.0,63.0,83.0,100.0,100.0,55.0,Abraxas,No alter egos found.,...,Cosmic Entity,"['-', '0 cm']","['- lb', '0 kg']",Blue,Black,Dimensional destroyer,-,Cosmic Beings,"Eternity (""Father"")",https://www.superherodb.com/pictures2/portrait...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
726,727,Yellowjacket II,50.0,10.0,35.0,28.0,31.0,28.0,Rita DeMara,No alter egos found.,...,Human,"[""5'5"", '165 cm']","['115 lb', '52 kg']",Blue,Strawberry Blond,"Adventurer; former criminal, electronics engineer",New York City area,"Formerly Guardians of the Galaxy, Avengers, Ma...",-,https://www.superherodb.com/pictures2/portrait...
727,728,Ymir,50.0,100.0,27.0,100.0,98.0,28.0,Ymir,No alter egos found.,...,Frost Giant,"['1000', '304.8 meters']","['- lb', '0 kg']",White,No Hair,-,Niffleheim,-,"Utgard-Loki, Loki, and the race of Frost Giant...",https://www.superherodb.com/pictures2/portrait...
728,729,Yoda,88.0,52.0,33.0,25.0,100.0,90.0,Yoda,No alter egos found.,...,Yoda's species,"[""2'2"", '66 cm']","['38 lb', '17 kg']",Brown,White,-,-,"Jedi Order, Jedi High Counsl, Galactic Republic","Master: N'Kata Del Gormo, Apprentices: Dooku, ...",https://www.superherodb.com/pictures2/portrait...
729,730,Zatanna,81.0,10.0,23.0,28.0,100.0,56.0,Zatanna Zatara,No alter egos found.,...,Human,"[""5'7"", '170 cm']","['127 lb', '57 kg']",Blue,Black,-,-,"Misty Kilgore, Seven Soldiers of Victory, Just...","Giovanni ""John"" Zatara (father, deceased), Sin...",https://www.superherodb.com/pictures2/portrait...


<h1>Analiza danych</h1>

In [4]:
origin_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 731 entries, 0 to 730
Data columns (total 26 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id                 731 non-null    int64  
 1   name               731 non-null    object 
 2   intelligence       566 non-null    float64
 3   strength           629 non-null    float64
 4   speed              566 non-null    float64
 5   durability         566 non-null    float64
 6   power              566 non-null    float64
 7   combat             566 non-null    float64
 8   full-name          630 non-null    object 
 9   alter-egos         731 non-null    object 
 10  aliases            731 non-null    object 
 11  place-of-birth     731 non-null    object 
 12  first-appearance   731 non-null    object 
 13  publisher          716 non-null    object 
 14  alignment          731 non-null    object 
 15  gender             731 non-null    object 
 16  race               429 non

Jak widzimy powyżej mamy kilka kolumn zawierających dane liczbowe które będą przydatne w dalszej analizie. Kolumna <u>alignment</u> posłuży nam jako szukana wartość przez model. Mamy również dwie kolumny <u>height</u> i <u>weight</u> z których bedziemy musieli wyekstaktować wartości liczbowe. Pozostawym również kolumny <u>name</u> oraz <u>publisher</u> do dalszej analizy

In [5]:
useful_columns = ['id', 'name', 'intelligence', 'strength', 'speed', 'durability', 'power', 'combat', 'publisher',
                  'alignment', 'gender', 'race', 'height', 'weight', 'eye-color', 'hair-color']
df = origin_df[useful_columns]

Wyekstraktujmy teraz <u>height</u>

In [6]:
df = extract_from_list_column(df, 'height', to_float_cm)

Oraz <u>weight</u>

In [7]:
df = extract_from_list_column(df, 'weight', to_float_kg)

Zobaczmy teraz czy w naszym zbiorze występują ubytki danych

In [8]:
df.isna().sum()

id                0
name              0
intelligence    165
strength        102
speed           165
durability      165
power           165
combat          165
publisher        15
alignment         0
gender            0
race            301
height            0
weight            0
eye-color         0
hair-color        0
dtype: int64

In [9]:
nan_rows = df[df.isna().any(axis=1)].shape[0]
all_rows = df.shape[0]
print(f"all_rows: {all_rows}, rows_with_nan: {nan_rows}, percent: {nan_rows / all_rows * 100}")

all_rows: 730, rows_with_nan: 336, percent: 46.02739726027397


Jak widzimy aż 46% obserwacji ma conajmniej jedną kolumne pustą i że najbardziej wybrakowana kolumna jest race. W kontekście poszukiwania przynależności bohatera do złych lub dobrych nie powinna miec ta kolumna statystycznego znaczenia wiec pozbądźmy sie jej

In [10]:
df = df.drop(columns=['race'])

In [11]:
nan_rows = df[df.isna().any(axis=1)].shape[0]
all_rows = df.shape[0]
print(f"all_rows: {all_rows}, rows_with_nan: {nan_rows}, percent: {nan_rows / all_rows * 100}")

all_rows: 730, rows_with_nan: 176, percent: 24.10958904109589


Jak widzimy dzięki tej operacji udało nam się prawie dwukrotnie zmniejszyć liczbę obserwacji z brakującymi danymi. Sprawdźmy teraz jaki procent rekordów ma n brakujących danych w kolumnach

In [12]:
print_nan_stat_in_row(df)

Column number with NaN: 0, Rows number: 730, Rows number with NaN: 554, percent: 75.89041095890411
Column number with NaN: 1, Rows number: 730, Rows number with NaN: 11, percent: 1.5068493150684932
Column number with NaN: 2, Rows number: 730, Rows number with NaN: 0, percent: 0.0
Column number with NaN: 3, Rows number: 730, Rows number with NaN: 0, percent: 0.0
Column number with NaN: 4, Rows number: 730, Rows number with NaN: 0, percent: 0.0
Column number with NaN: 5, Rows number: 730, Rows number with NaN: 60, percent: 8.21917808219178
Column number with NaN: 6, Rows number: 730, Rows number with NaN: 104, percent: 14.246575342465754
Column number with NaN: 7, Rows number: 730, Rows number with NaN: 1, percent: 0.136986301369863
Column number with NaN: 8, Rows number: 730, Rows number with NaN: 0, percent: 0.0
Column number with NaN: 9, Rows number: 730, Rows number with NaN: 0, percent: 0.0
Column number with NaN: 10, Rows number: 730, Rows number with NaN: 0, percent: 0.0
Column nu

Jak widać mamy sporo rekordów gdzie więcej niż 5 kolumn ma brakującą wartość, usunmy te rekordy poniewaz taka ilość cech dla rekordu ciężko będzie w sensowny sposób uzupełnić.

In [13]:
df = df[df.isna().sum(axis=1) < 5]

In [14]:
print_nan_stat_in_row(df)

Column number with NaN: 0, Rows number: 565, Rows number with NaN: 554, percent: 98.05309734513274
Column number with NaN: 1, Rows number: 565, Rows number with NaN: 11, percent: 1.9469026548672566
Column number with NaN: 2, Rows number: 565, Rows number with NaN: 0, percent: 0.0
Column number with NaN: 3, Rows number: 565, Rows number with NaN: 0, percent: 0.0
Column number with NaN: 4, Rows number: 565, Rows number with NaN: 0, percent: 0.0
Column number with NaN: 5, Rows number: 565, Rows number with NaN: 0, percent: 0.0
Column number with NaN: 6, Rows number: 565, Rows number with NaN: 0, percent: 0.0
Column number with NaN: 7, Rows number: 565, Rows number with NaN: 0, percent: 0.0
Column number with NaN: 8, Rows number: 565, Rows number with NaN: 0, percent: 0.0
Column number with NaN: 9, Rows number: 565, Rows number with NaN: 0, percent: 0.0
Column number with NaN: 10, Rows number: 565, Rows number with NaN: 0, percent: 0.0
Column number with NaN: 11, Rows number: 565, Rows num

Jak widać zostało nam 11 rekordów z brakującymi danymi

In [15]:
df[df.isna().any(axis=1)]

,id,name,intelligence,strength,speed,durability,power,combat,publisher,alignment,gender,height,weight,eye-color,hair-color
86,87,Bionic Woman,56.0,37.0,33.0,40.0,20.0,40.0,NaN,good,Female,0.0,0.0,Blue,Black
138,139,Brundlefly,69.0,32.0,25.0,40.0,27.0,15.0,NaN,-,Male,193.0,0.0,-,-
175,176,Chuck Norris,50.0,80.0,47.0,56.0,42.0,99.0,NaN,good,Male,178.0,0.0,-,-
244,245,Ethan Hunt,75.0,11.0,29.0,30.0,26.0,95.0,NaN,good,Male,168.0,0.0,Brown,Brown
286,287,Godzilla,44.0,100.0,54.0,100.0,100.0,20.0,NaN,bad,-,10800.0,90000000.0,-,-
354,355,Jason Bourne,88.0,12.0,29.0,30.0,26.0,100.0,NaN,good,Male,0.0,0.0,-,-
380,381,Katniss Everdeen,56.0,8.0,21.0,25.0,24.0,55.0,NaN,good,Female,0.0,0.0,-,-
388,389,King Kong,56.0,100.0,71.0,75.0,47.0,75.0,NaN,good,Male,3050.0,9000000.0,Yellow,Black
392,393,Kool-Aid Man,25.0,18.0,8.0,10.0,9.0,14.0,NaN,good,Male,0.0,0.0,Black,No Hair
539,540,Rambo,63.0,14.0,25.0,30.0,30.0,100.0,NaN,good,Male,178.0,83.0,Brown,Black


We wszystkich rekordach brakuje wartości <u>publisher</u>, analizując imona bohaterów widzimy że w większości nie są to bohaterowie komiksowi. Wstawmy więc w kolumnę <u>publisher</u> wartość <b>Other</b>

In [16]:
df.loc[df.isna().any(axis=1), 'publisher'] = 'Other'

W nastepnym kroku sprawdźmy czy mamy wartości odstające dla kolumn numerycznych

In [17]:
numeric_columns = df.select_dtypes(include=np.number).columns.tolist()
numeric_columns.remove('id')
row_num_before = df.shape[0]
for col in numeric_columns:
    outliers_num = calc_outliers(df, col)
    print(f'column: {col}, outliers number: {outliers_num}\n')


column: intelligence, outliers number: 0

column: strength, outliers number: 0

column: speed, outliers number: 0

column: durability, outliers number: 0

column: power, outliers number: 0

column: combat, outliers number: 0

column: height, outliers number: 4

column: weight, outliers number: 1



Usunmy je

In [18]:
for col in ['height', 'weight']:
    df = remove_outliers(df, col)
    outliers_num = calc_outliers(df, col)
    print(f'col: {col}, outliers: {outliers_num}\n')

col: height, outliers: 5

col: weight, outliers: 3



Kolumny <u>name</u> nie będziemy używać w modelowaniu. Usuńmy ją

In [19]:
df = df.drop(columns=['name'])
df.columns

Index(['id', 'intelligence', 'strength', 'speed', 'durability', 'power',
       'combat', 'publisher', 'alignment', 'gender', 'height', 'weight',
       'eye-color', 'hair-color'],
      dtype='object')

W data frame mamy dane tekstowe które mogą zostać zakodowane jako numeryczne kategorie. Zakodujmy je

In [20]:
string_features = df.select_dtypes(include=np.dtype).columns.tolist()
encoders_by_column  = {}

for col in string_features:
    encoder = OrdinalEncoder()
    df[col] = encoder.fit_transform(df[[col]])
    encoders_by_column[col] = encoder

Przygotowany dataset zapiszmy do pliku

In [21]:
df.to_csv(path / 'data' / 'prepared_dataset.csv', index=False)

Zapiszmy również konfiguracje

In [22]:
config_path = path / 'configuration' / 'config.json'
config = load_config(config_path)

config.prediction_column = 'alignment'
config.features_names = df.drop(columns=['id', config.prediction_column]).columns.values.tolist()
config.prediction_categories = encoders_by_column[config.prediction_column].categories_[0].tolist()

config.save_to_file(config_path)